# Caso 0 — Cervejaria Beerlink (Google OR-Tools)
**Treinamento de Otimização · Genoa para Gradus · M1 — PL e o Simplex**

Resolução do mesmo modelo do Excel, agora em Python com [Google OR-Tools](https://developers.google.com/optimization). 
Use este notebook como apoio: ele mostra como o modelo fica num formato versionável (Git) e fácil de automatizar.

**Dica:** rode no [Google Colab](https://colab.research.google.com/) — não precisa instalar nada localmente.


In [ ]:
# Instala o OR-Tools (no Colab basta a primeira vez de cada sessão)
!pip install ortools -q

## 1) Parâmetros
Mesmos números que estão no bloco **PARÂMETROS** da planilha `caso0_cervejaria_sol.xlsx`.

In [ ]:
# Recursos disponíveis por semana
MALTE       = 100   # kg
LUPULO      = 80    # g
FERMENTADOR = 200   # h

# Coeficientes técnicos (consumo por litro)
consumo = {
    'IPA':    {'malte': 0.8, 'lupulo': 0.5, 'ferm': 1.2},
    'Pilsen': {'malte': 0.6, 'lupulo': 0.4, 'ferm': 0.8},
}

# Margem de contribuição (R$/L)
margem = {'IPA': 15, 'Pilsen': 12}

# Demanda mínima (contrato)
DEMANDA_MIN_IPA = 50  # L/sem

## 2) Modelo PL com OR-Tools
Para problemas lineares contínuos (sem variáveis inteiras), o solver indicado é o **GLOP** (Google's Linear Programming solver). Se houver variáveis inteiras, troca-se para **CBC** ou **SCIP**.

$\max\ z = 15\, x_{I} + 12\, x_{P}$

In [ ]:
from ortools.linear_solver import pywraplp

# 1) Cria solver
solver = pywraplp.Solver.CreateSolver('GLOP')
assert solver, 'Solver não disponível.'

# 2) Variáveis (contínuas, ≥ 0)
xI = solver.NumVar(0, solver.infinity(), 'IPA')
xP = solver.NumVar(0, solver.infinity(), 'Pilsen')

# 3) Restrições
solver.Add(consumo['IPA']['malte']  * xI + consumo['Pilsen']['malte']  * xP <= MALTE)
solver.Add(consumo['IPA']['lupulo'] * xI + consumo['Pilsen']['lupulo'] * xP <= LUPULO)
solver.Add(consumo['IPA']['ferm']   * xI + consumo['Pilsen']['ferm']   * xP <= FERMENTADOR)
solver.Add(xI >= DEMANDA_MIN_IPA)

# 4) Função objetivo (maximizar lucro)
solver.Maximize(margem['IPA'] * xI + margem['Pilsen'] * xP)

print(f'Variáveis : {solver.NumVariables()}')
print(f'Restrições: {solver.NumConstraints()}')

## 3) Solução

In [ ]:
status = solver.Solve()

STATUS = {
    pywraplp.Solver.OPTIMAL:     'OPTIMAL',
    pywraplp.Solver.FEASIBLE:    'FEASIBLE',
    pywraplp.Solver.INFEASIBLE:  'INFEASIBLE',
    pywraplp.Solver.UNBOUNDED:   'UNBOUNDED',
    pywraplp.Solver.ABNORMAL:    'ABNORMAL',
    pywraplp.Solver.NOT_SOLVED:  'NOT_SOLVED',
}
print(f'Status: {STATUS[status]}')

if status == pywraplp.Solver.OPTIMAL:
    print(f'IPA    = {xI.solution_value():.1f} L')
    print(f'Pilsen = {xP.solution_value():.1f} L')
    lucro = solver.Objective().Value()
    print(f'Lucro  = R$ {lucro:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.'))

**Saída esperada:** `OPTIMAL · IPA = 50 L · Pilsen = 100 L · Lucro = R$ 1.950,00` — idêntico ao Excel.

## 4) Análise de sensibilidade — preços-sombra (dual values)
O preço-sombra mostra quanto a FO melhoraria se o lado direito da restrição aumentasse em 1 unidade. Útil para responder: *"Vale a pena pagar R$ X por kg adicional de malte?"*

No OR-Tools / GLOP, os duais são acessados pelo método `.dual_value()` das restrições.

In [ ]:
# Recriar o modelo guardando referência das restrições, para extrair os duais
solver = pywraplp.Solver.CreateSolver('GLOP')
xI = solver.NumVar(0, solver.infinity(), 'IPA')
xP = solver.NumVar(0, solver.infinity(), 'Pilsen')

c_malte  = solver.Add(0.8 * xI + 0.6 * xP <= 100, 'malte')
c_lupulo = solver.Add(0.5 * xI + 0.4 * xP <= 80,  'lupulo')
c_ferm   = solver.Add(1.2 * xI + 0.8 * xP <= 200, 'fermentador')
c_contr  = solver.Add(xI >= 50,                   'contrato_IPA')
solver.Maximize(15 * xI + 12 * xP)

solver.Solve()

for nome, c in [('malte', c_malte), ('lupulo', c_lupulo), ('fermentador', c_ferm), ('contrato_IPA', c_contr)]:
    sombra = c.dual_value()
    print(f'{nome:>14s}  preço-sombra = {sombra:>8.3f}')

**Como ler:**
- `malte` está apertada com preço-sombra ≈ 20 → cada kg adicional de malte traz +R$ 20 de lucro. Vale pagar até R$ 20/kg por malte extra.
- `lupulo` e `fermentador` têm folga (preço-sombra = 0) → não adianta comprar mais.
- `contrato_IPA` tem preço-sombra **negativo** → o contrato com o bar parceiro está *custando* dinheiro a Maurício. Se ele pudesse renegociar, ganharia mais produzindo só Pilsen.

## 5) Quando sair do Excel?
Excel resolve este caso instantaneamente. Mas se Maurício escalar:
- 50 cervejas em 12 cervejarias → 600 variáveis. Solver padrão trava em 200.
- Quer rodar 100 cenários (preço da uva, demanda) → automatizar com Python é trivial; no Excel é macro.
- Quer expor a otimização num app web → FastAPI + OR-Tools é direto.

Veja `m3_gmc.ipynb` para um caso onde o Excel realmente começa a apertar (programação inteira mista + linearização por trechos).